# 03 — Model

**Cel notebooka:** zbudować model, który przewiduje wynik GP Meksyku 2026 (zwycięzca, top 3, top 10, najszybsze okrążenie), i uczciwie sprawdzić, czy jest lepszy od reguły „wygra pole position” (72% trafień).

Wczytujemy wyłącznie CSV z `01_dane`. Zmienne liczymy tylko z danych **sprzed** przewidywanego wyścigu, żeby nie dopuścić do wycieku danych.

## 3.1 Tabela bazowa: jeden wiersz = kierowca w wyścigu

**Co robimy:** z kwalifikacji i wyników budujemy jedną tabelę, w której każdy wiersz to jeden kierowca w jednym wyścigu (rundy 1–19, razem z Meksykiem).

**Dlaczego:** modele uczą się z tabeli, w której wiersz to jedna obserwacja. Meksyk (runda 19) też tu trafia, ale z pustym wynikiem (`race_pos`), bo właśnie tego szukamy. Dzięki temu cechy dla Meksyku policzymy tym samym kodem co dla wyścigów treningowych.

**Uwaga:** w tabeli są braki: kilku kierowców nie ma pozycji w kwalifikacjach (wiemy z 2.5), a DNF-y nie mają pozycji na mecie. Zajmiemy się nimi przy budowie cech.

In [19]:
import pandas as pd

quali = pd.read_csv("data/quali_2026.csv")
results = pd.read_csv("data/results_2026.csv")

drivers = pd.concat([
    results[["round", "name_acronym", "team_name"]],
    quali[["round", "name_acronym", "team_name"]],
]).drop_duplicates(["round", "name_acronym"])

q = quali[["round", "name_acronym", "position", "best_time", "gap_to_pole_pct"]].rename(columns={"position": "quali_pos"})
r = results[["round", "name_acronym", "position", "dnf", "dns"]].rename(columns={"position": "race_pos"})

base = drivers.merge(q, on=["round", "name_acronym"], how="left").merge(r, on=["round", "name_acronym"], how="left")
base = base.sort_values(["round", "name_acronym"]).reset_index(drop=True)

print(base.shape)
print(base.groupby("round").size().unique())
base.isna().sum()

(418, 9)
[22]


round               0
name_acronym        0
team_name           0
quali_pos           8
best_time           6
gap_to_pole_pct     6
race_pos           96
dnf                22
dns                22
dtype: int64

## 3.2 Punkt odniesienia: „kolejność jak na starcie”

**Co robimy:** oceniamy najprostszą możliwą prognozę — meta będzie taka sama jak start. Zwycięzca = pole position, podium = trójka z przodu, top 10 = dziesiątka z przodu.

**Dlaczego:** każdy model musi być lepszy od tej reguły, inaczej jest niepotrzebnie skomplikowany. Wynik liczymy na wyścigach 1–18, których wynik znamy.

**Jak mierzymy** (osobno dla każdego wyścigu, potem średnia):
- **zwycięzca:** trafiony albo nie,
- **top 3:** ile z 3 przewidzianych nazwisk faktycznie było na podium (kolejności nie liczymy),
- **top 10:** ile z 10 przewidzianych było w pierwszej dziesiątce.

Kierowcy bez pozycji w kwalifikacjach dostają 22 (koniec stawki).

In [20]:
base["start_pos"] = base["quali_pos"].fillna(22)

def score_race(g, pred_col):
    """Trafienia dla jednego wyścigu: zwycięzca, ile z 3 / 10 przewidzianych miejsc faktycznie w top 3 / top 10."""
    order = g.sort_values(pred_col)
    actual = g[g["race_pos"].notna()]
    return pd.Series({
        "winner_hit": order["name_acronym"].iloc[0] == actual.loc[actual["race_pos"] == 1, "name_acronym"].iloc[0],
        "top3_overlap": len(set(order["name_acronym"].iloc[:3]) & set(actual.loc[actual["race_pos"] <= 3, "name_acronym"])) / 3,
        "top10_overlap": len(set(order["name_acronym"].iloc[:10]) & set(actual.loc[actual["race_pos"] <= 10, "name_acronym"])) / 10,
    })

done = base[base["round"] <= 18]
baseline = done.groupby("round").apply(score_race, pred_col="start_pos", include_groups=False)

print(f"Zwycięzca: {baseline['winner_hit'].mean():.0%} | top 3: {baseline['top3_overlap'].mean():.0%} | top 10: {baseline['top10_overlap'].mean():.0%}")
baseline

Zwycięzca: 72% | top 3: 61% | top 10: 77%


,winner_hit,top3_overlap,top10_overlap
round,,,
1,True,0.666667,0.7
2,True,1.000000,0.7
3,True,0.666667,0.7
4,True,0.333333,0.8
5,False,0.333333,0.6
6,True,0.666667,0.6
7,False,0.666667,0.7
8,True,0.333333,1.0
9,False,0.666667,0.7


## 3.3 Zmienna: umiejętność kierowcy (zysk miejsc ponad typowy)

**Co robimy:** dla każdego kierowcy i każdej rundy liczymy, ile miejsc **ponad typowy dla pozycji startowej** zyskiwał w *poprzednich* wyścigach (tak jak w 2.4), ale ostrożnie:
- **tylko z wcześniejszych rund** — dla rundy 10 używamy rund 1–9, a nie rund z przyszłości (to zapobiega wyciekowi danych),
- **ze skurczeniem:** sumę dzielimy przez (liczba wyścigów + 5), a nie przez liczbę wyścigów. Kierowca z jednym dobrym wyścigiem nie dostaje wtedy dużej wartości, a przy wielu wyścigach skurczenie jest coraz mniejsze.

**Dlaczego:** w 2.4 widzieliśmy, że to słaby i zaszumiony sygnał. Skurczenie ogranicza szum, a liczenie tylko z przeszłości sprawia, że model uczy się na tym, co faktycznie wiedzielibyśmy przed wyścigiem. Kierowca bez historii dostaje 0 (neutralnie).

**Uwaga:** wyścigi, w których kierowca odpadł (`dnf`), pomijamy — także te, w których został sklasyfikowany na końcu stawki, bo przejechał prawie cały dystans. Awaria to nie brak tempa; awaryjność liczymy osobno w 3.4.

In [21]:
base["gain"] = (base["start_pos"] - base["race_pos"]).where(~base["dnf"].eq(True))
K = 5

def driver_skill(base, k=K):
    """Dla każdej rundy: skurczona średnia 'zysku ponad typowy' kierowcy, liczona tylko z wcześniejszych rund."""
    out = pd.Series(0.0, index=base.index)
    for rnd in sorted(base["round"].unique()):
        past = base[(base["round"] < rnd) & base["gain"].notna()].copy()
        past["excess"] = past["gain"] - past.groupby("start_pos")["gain"].transform("mean")
        agg = past.groupby("name_acronym")["excess"].agg(["sum", "count"])
        skill = agg["sum"] / (agg["count"] + k)
        rows = base["round"] == rnd
        out[rows] = base.loc[rows, "name_acronym"].map(skill).fillna(0).values
    return out

base["skill"] = driver_skill(base)
base[base["round"] == 19].sort_values("skill", ascending=False)[["name_acronym", "team_name", "start_pos", "skill"]].round(2)

,name_acronym,team_name,start_pos,skill
417,VER,Red Bull Racing,4.0,1.60
398,ANT,Mercedes,5.0,0.83
408,LAW,Racing Bulls,22.0,0.70
404,HAD,Red Bull Racing,7.0,0.40
411,NOR,McLaren,6.0,0.38
407,HUL,Audi,10.0,0.35
405,HAM,Ferrari,1.0,0.32
409,LEC,Ferrari,3.0,0.14
410,LIN,Racing Bulls,8.0,0.13
402,COL,Alpine,11.0,0.09


## 3.4 Zmienna: awaryjność (kierowca i zespół)

**Co robimy:** dla każdej rundy liczymy odsetek wyścigów, w których kierowca (i osobno zespół) **nie dojechał lub nie wystartował**, ale tylko z poprzednich rund i ze skurczeniem.

**Dlaczego:** w 2.7 awarie częściowo mieszały kolejność, więc dla modelu liczy się szansa, że ktoś odpadnie. Liczymy ją dwa razy: dla zespołu (ma więcej danych: dwa samochody co wyścig, a awarie bywają techniczne) i dla kierowcy (np. wypadki).

**Skurczenie:** zamiast surowego odsetka bierzemy `(awarie + k · średnia ogólna) / (wyścigi + k)`. Przy małej liczbie wyścigów wynik zbliża się do średniej całej stawki (około 20%), a przy większej do własnego odsetka. Dla kierowcy `k = 5`, dla zespołu `k = 10` (zespół ma dwa razy więcej danych na wyścig, więc potrzebuje mniejszego „ciężaru” średniej).

In [22]:
import numpy as np

known = base["dnf"].notna()
base["out"] = np.where(known, base["dnf"].eq(True) | base["dns"].eq(True), np.nan)

def shrunk_rate(base, group_col, k):
    """Skurczony odsetek 'nie dojechał / nie wystartował' dla grupy, liczony tylko z wcześniejszych rund."""
    out = pd.Series(0.0, index=base.index)
    for rnd in sorted(base["round"].unique()):
        past = base[(base["round"] < rnd) & base["out"].notna()]
        prior = past["out"].mean() if len(past) else 0.1
        agg = past.groupby(group_col)["out"].agg(["sum", "count"])
        rate = (agg["sum"] + k * prior) / (agg["count"] + k)
        rows = base["round"] == rnd
        out[rows] = base.loc[rows, group_col].map(rate).fillna(prior).values
    return out

base["driver_out_rate"] = shrunk_rate(base, "name_acronym", 5)
base["team_out_rate"] = shrunk_rate(base, "team_name", 10)

base[base["round"] == 19].drop_duplicates("team_name").sort_values("team_out_rate")[["team_name", "team_out_rate"]].round(2)

,team_name,team_out_rate
408,Racing Bulls,0.09
398,Mercedes,0.13
405,Ferrari,0.13
402,Alpine,0.15
399,Haas F1 Team,0.17
400,Audi,0.17
404,Red Bull Racing,0.17
411,McLaren,0.20
396,Williams,0.26
401,Cadillac,0.35


## 3.5 Walidacja walk-forward i pierwszy model (tylko pozycja startowa)

**Co robimy:** budujemy narzędzie do uczciwego oceniania modeli, a potem testujemy je na najprostszym modelu — regresji Ridge, która ma tylko pozycję startową.

**Walk-forward:** dla każdej rundy `R` (od 4. do 18.) uczymy model **wyłącznie na rundach przed R**, przewidujemy kolejność w rundzie R i porównujemy z prawdą. To symuluje prawdziwe użycie: w dniu wyścigu znamy tylko przeszłość. Pierwsze trzy rundy służą wyłącznie do nauki.

**Model:** Ridge to regresja liniowa z karą za zbyt duże współczynniki (zapobiega przesadnemu dopasowaniu). Uczymy go na kierowcach, którzy **ukończyli** wyścig (`gain` niepuste) i przewidujemy pozycję na mecie. Potem sortujemy kierowców po przewidzianej pozycji i oceniamy jak w 3.2.

**Test poprawności:** model z jedną zmienną (pozycją startową) musi dać dokładnie to samo co reguła „jak na starcie”, bo im większa pozycja startowa, tym gorsza przewidywana meta. Jeśli wyniki się zgadzają, narzędzie działa.

In [23]:
from sklearn.linear_model import Ridge

def walk_forward(features, first_round=4, alpha=1.0):
    """Dla każdej rundy od `first_round`: uczymy model na wcześniejszych rundach i oceniamy go na tej jednej."""
    rows = []
    for rnd in range(first_round, 19):
        train = base[(base["round"] < rnd) & base["gain"].notna()]
        test = base[base["round"] == rnd].copy()
        model = Ridge(alpha=alpha).fit(train[features], train["race_pos"])
        test["pred"] = model.predict(test[features])
        rows.append(score_race(test, "pred").rename(rnd))
    return pd.DataFrame(rows)

def summary(scores):
    return f"Zwycięzca: {scores['winner_hit'].mean():.0%} | top 3: {scores['top3_overlap'].mean():.0%} | top 10: {scores['top10_overlap'].mean():.0%}"

same_rounds = baseline.loc[4:18]
print("Reguła 'jak na starcie' (rundy 4–18):", summary(same_rounds))
print("Ridge, tylko pozycja startowa:        ", summary(walk_forward(["start_pos"])))

Reguła 'jak na starcie' (rundy 4–18): Zwycięzca: 67% | top 3: 58% | top 10: 78%
Ridge, tylko pozycja startowa:         Zwycięzca: 67% | top 3: 58% | top 10: 78%


## 3.6 Czy dodatkowe zmienne pomagają?

**Co robimy:** porównujemy walk-forward kilka zestawów zmiennych: sama pozycja startowa, plus strata do pole (`gap`, w % — dla brakujących kwalifikacji wstawiamy największą stratę w rundzie), plus `skill`, plus awaryjność i wszystko razem.

**Dlaczego:** zasada z EDA — zmienna zostaje tylko wtedy, gdy poprawia wynik na wyścigach, których model nie widział.

**Uwaga o skali:** 15 wyścigów to mało. Jeden wyścig to ok. 6,7 punktu procentowego trafienia zwycięzcy, więc różnica jednego wyścigu to szum, nie dowód.

In [24]:
base["gap"] = base["gap_to_pole_pct"].fillna(base.groupby("round")["gap_to_pole_pct"].transform("max"))

feature_sets = {
    "start_pos": ["start_pos"],
    "+ gap": ["start_pos", "gap"],
    "+ skill": ["start_pos", "skill"],
    "+ awaryjność": ["start_pos", "driver_out_rate", "team_out_rate"],
    "wszystko": ["start_pos", "gap", "skill", "driver_out_rate", "team_out_rate"],
}

print("Reguła 'jak na starcie':".ljust(24), summary(same_rounds))
for name, feats in feature_sets.items():
    print(name.ljust(24), summary(walk_forward(feats)))

Reguła 'jak na starcie': Zwycięzca: 67% | top 3: 58% | top 10: 78%
start_pos                Zwycięzca: 67% | top 3: 58% | top 10: 78%
+ gap                    Zwycięzca: 67% | top 3: 58% | top 10: 78%
+ skill                  Zwycięzca: 60% | top 3: 58% | top 10: 78%
+ awaryjność             Zwycięzca: 67% | top 3: 58% | top 10: 78%
wszystko                 Zwycięzca: 67% | top 3: 56% | top 10: 79%


## 3.7 Ciągłe miary błędu

**Co robimy:** oceniamy te same zestawy zmiennych dokładniejszymi miarami niż „trafiony zwycięzca”:
- **średni błąd pozycji** — o ile miejsc średnio myli się model (w kierowcach, którzy ukończyli wyścig),
- **korelacja Spearmana** między przewidzianą a faktyczną kolejnością.

**Dlaczego:** trafienie zwycięzcy czy podium zmienia się skokowo (jeden wyścig = 6,7 pp) i ignoruje drobne poprawy. Ciągłe miary reagują na każdą zmianę, więc jeśli któraś zmienna cokolwiek wnosi, tu to zobaczymy.

In [25]:
def walk_forward_error(features, first_round=4, alpha=1.0):
    """Jak walk_forward, ale mierzy ciągłe błędy: średni błąd pozycji (w miejscach) i korelację Spearmana z metą."""
    rows = []
    for rnd in range(first_round, 19):
        train = base[(base["round"] < rnd) & base["gain"].notna()]
        test = base[(base["round"] == rnd) & base["gain"].notna()].copy()
        model = Ridge(alpha=alpha).fit(train[features], train["race_pos"])
        test["pred"] = model.predict(test[features])
        rows.append({
            "round": rnd,
            "mae": (test["pred"] - test["race_pos"]).abs().mean(),
            "spearman": test["pred"].corr(test["race_pos"], method="spearman"),
        })
    return pd.DataFrame(rows).set_index("round")

print("Zestaw".ljust(16), "błąd (miejsca)", " Spearman")
for name, feats in feature_sets.items():
    err = walk_forward_error(feats)
    print(name.ljust(16), f"{err['mae'].mean():14.2f}", f"{err['spearman'].mean():9.3f}")

Zestaw           błąd (miejsca)  Spearman
start_pos                  2.20     0.846
+ gap                      2.21     0.847
+ skill                    2.23     0.845
+ awaryjność               2.20     0.846
wszystko                   2.23     0.847


## 3.8 Tempo wyścigowe z okrążeń

**Co robimy:** z zapisanych okrążeń (`data/raw/laps_*`, bez nowych zapytań do API) liczymy dla każdego kierowcy i wyścigu **typowe tempo na dystansie**, wyrażone jako % względem środka stawki. Wartość ujemna = szybciej niż typowo, dodatnia = wolniej.

**Dlaczego:** kwalifikacje mierzą jedno szybkie okrążenie, a o wyścigu decyduje tempo na dystansie. Kierowca może startować z P5, a jechać jak z P1 (np. ANT). To informacja, której nie ma w pozycji startowej.

**Które okrążenia liczymy (czyszczenie):**
- bez 1. okrążenia (start, tłok),
- bez okrążeń wyjazdowych z alei i zjazdowych do alei (zjazd jest wolny, wyjazd na zimnych oponach),
- bez okrążeń za samochodem bezpieczeństwa (czas dłuższy niż 107% mediany wyścigu),
- kierowcy z mniej niż 10 czystymi okrążeniami odpadają (za mało danych).

Tempo kierowcy = **mediana** jego czystych okrążeń (odporna na pojedyncze skrajne wartości), a potem dzielimy przez medianę tych median w wyścigu.

In [26]:
import json
from pathlib import Path

races = pd.read_csv("data/races_2026.csv")
rows = []
for race in races[races["round"] <= 18].itertuples():
    laps = pd.DataFrame(json.loads(Path(f"data/raw/laps_session_key={race.session_key}.json").read_text()))
    laps = laps.sort_values(["driver_number", "lap_number"])
    # okrążenie zjazdowe = tuż przed okrążeniem wyjazdowym z alei
    laps["is_in_lap"] = laps.groupby("driver_number")["is_pit_out_lap"].shift(-1, fill_value=False)
    clean = laps[(laps["lap_number"] > 1) & ~laps["is_pit_out_lap"] & ~laps["is_in_lap"] & laps["lap_duration"].notna()]
    clean = clean[clean["lap_duration"] < 1.07 * clean["lap_duration"].median()]   # odrzucamy okrążenia za samochodem bezpieczeństwa
    per_driver = clean.groupby("driver_number")["lap_duration"].agg(["median", "count"])
    per_driver = per_driver[per_driver["count"] >= 10]
    per_driver["pace_pct"] = (per_driver["median"] / per_driver["median"].median() - 1) * 100
    per_driver["round"] = race.round
    rows.append(per_driver.reset_index()[["round", "driver_number", "pace_pct"]])

pace = pd.concat(rows).merge(results[["round", "driver_number", "name_acronym"]], on=["round", "driver_number"])
pace = pace[["round", "name_acronym", "pace_pct"]]
print(pace.shape)
pace[pace["round"] == 18].sort_values("pace_pct").round(2)

(368, 3)


,round,name_acronym,pace_pct
352,18,ANT,-1.86
346,18,NOR,-1.65
347,18,VER,-1.61
354,18,LEC,-1.54
362,18,HAM,-1.39
366,18,PIA,-0.96
364,18,RUS,-0.87
349,18,HAD,-0.78
350,18,GAS,-0.65
358,18,LAW,-0.56


## 3.9 Tempo z poprzednich wyścigów jako zmienna

**Co robimy:** zamieniamy tempo z 3.8 na zmienne dla modelu — `driver_pace` (kierowca) i `team_pace` (zespół) — i sprawdzamy w walk-forward, czy poprawiają przewidywanie.

**Jak liczymy:** tak jak `skill`: dla rundy R bierzemy tylko rundy przed R (tempo z tego samego wyścigu to wyciek danych), sumę dzielimy przez (liczba wyścigów + k), więc przy małej próbie wynik jest bliżej zera. Dla kierowcy `k = 2`, dla zespołu `k = 4`. Brak historii = 0 (neutralnie).

**Dlaczego dwie zmienne:** tempo zespołu to szybkość samochodu (ma więcej danych: dwa samochody co wyścig), a tempo kierowcy to jego własna forma w tych samochodach.

In [27]:
pace_team = pace.merge(base[["round", "name_acronym", "team_name"]], on=["round", "name_acronym"])

def past_pace(base, pace_team, group_col, k):
    """Skurczona średnia tempa (%) grupy z wcześniejszych rund; brak historii = 0 (neutralnie)."""
    out = pd.Series(0.0, index=base.index)
    for rnd in sorted(base["round"].unique()):
        past = pace_team[pace_team["round"] < rnd]
        agg = past.groupby(group_col)["pace_pct"].agg(["sum", "count"])
        rows = base["round"] == rnd
        out[rows] = base.loc[rows, group_col].map(agg["sum"] / (agg["count"] + k)).fillna(0).values
    return out

base["driver_pace"] = past_pace(base, pace_team, "name_acronym", 2)
base["team_pace"] = past_pace(base, pace_team, "team_name", 4)

feature_sets_pace = {
    "start_pos": ["start_pos"],
    "+ tempo kierowcy": ["start_pos", "driver_pace"],
    "+ tempo zespołu": ["start_pos", "team_pace"],
    "+ oba tempa": ["start_pos", "driver_pace", "team_pace"],
}

print("Zestaw".ljust(18), "zwycięzca  top 3  top 10 | błąd   Spearman")
for name, feats in feature_sets_pace.items():
    s, e = walk_forward(feats), walk_forward_error(feats)
    print(name.ljust(18), f"{s['winner_hit'].mean():8.0%} {s['top3_overlap'].mean():7.0%} {s['top10_overlap'].mean():6.0%} | {e['mae'].mean():5.2f} {e['spearman'].mean():9.3f}")

Zestaw             zwycięzca  top 3  top 10 | błąd   Spearman
start_pos               67%     58%    78% |  2.20     0.846
+ tempo kierowcy        73%     62%    78% |  2.06     0.849
+ tempo zespołu         60%     62%    78% |  2.08     0.847
+ oba tempa             73%     62%    78% |  2.07     0.848


## 3.10 Model końcowy i przewidywanie dla Meksyku

**Co robimy:** uczymy model (Ridge ze zmiennymi `start_pos` i `driver_pace`) na **wszystkich 18 zakończonych wyścigach** i przewidujemy pozycję każdego kierowcy w Meksyku. Potem układamy kierowców po przewidzianej pozycji.

**Dlaczego te dwie zmienne:** to jedyny zestaw z 3.9, który poprawił wszystkie miary w walk-forward. Pozostałe zmienne (`gap`, `skill`, awaryjność, tempo zespołu) niczego nie wnosiły, więc ich nie używamy — prostszy model jest bezpieczniejszy.

**Jak czytać wynik:** `pred_pos` to przewidziana pozycja na mecie w sensie średnim (np. 2,3 nie oznacza P2,3, tylko że kierowca jest przewidywany wyżej niż ktoś z 3,0). Zawsze ściągana w stronę środka stawki — liczy się kolejność, nie sama liczba.

In [28]:
FEATURES = ["start_pos", "driver_pace"]

train = base[(base["round"] <= 18) & base["gain"].notna()]
final_model = Ridge(alpha=1.0).fit(train[FEATURES], train["race_pos"])
print("Współczynniki:", dict(zip(FEATURES, final_model.coef_.round(2))), "| wyraz wolny:", round(final_model.intercept_, 2))

mexico = base[base["round"] == 19].copy()
mexico["pred_pos"] = final_model.predict(mexico[FEATURES])
mexico = mexico.sort_values("pred_pos").reset_index(drop=True)
mexico.index += 1
mexico[["name_acronym", "team_name", "start_pos", "driver_pace", "pred_pos"]].round(2)

Współczynniki: {'start_pos': np.float64(0.53), 'driver_pace': np.float64(1.43)} | wyraz wolny: 3.8


,name_acronym,team_name,start_pos,driver_pace,pred_pos
1,HAM,Ferrari,1.0,-1.42,2.30
2,RUS,Mercedes,2.0,-1.30,3.00
3,LEC,Ferrari,3.0,-1.55,3.17
4,ANT,Mercedes,5.0,-1.76,3.95
5,VER,Red Bull Racing,4.0,-1.33,4.03
6,NOR,McLaren,6.0,-1.21,5.27
7,HAD,Red Bull Racing,7.0,-0.81,6.36
8,LIN,Racing Bulls,8.0,0.10,8.21
9,GAS,Alpine,9.0,-0.22,8.28
10,PIA,McLaren,12.0,-1.01,8.75


## 3.11 Zapis przewidywania modelu

**Co robimy:** zapisujemy tabelę z przewidzianymi pozycjami do CSV.

**Dlaczego:** `04_prediction` zaczyna od wczytania tego pliku, bez ponownego liczenia modelu. Dzięki temu prognoza jest „zamrożona”: nie zmieni się, gdy po wyścigu pojawią się nowe dane.

In [29]:
mexico[["name_acronym", "team_name", "start_pos", "driver_pace", "pred_pos"]].to_csv("data/model_prediction_mexico.csv", index=False)
print("Zapisano data/model_prediction_mexico.csv:", len(mexico), "kierowców")

Zapisano data/model_prediction_mexico.csv: 22 kierowców
